# Regulation Extraction : From text to JSON

This code extracting regulations from text file to JSON in hierarchical tree. This code uses Thai Capital market regulation

In [10]:
INPUT_FILE = 'raw_regulations/sec_reg.txt'
OUTPUT_FILE = 'intemediate_data/sec_reg_structured.json'

In [11]:
import re
import json

In [12]:

def process_hierarchy(text, patterns, current_parents=None):
    if current_parents is None:
        current_parents = []
        
    if not patterns:
        return []
    
    regex, current_type = patterns[0]
    chunks = re.split(regex, text.strip())
    
    if len(chunks) == 1:
        return process_hierarchy(text, patterns[1:], current_parents)
    
    subsections = []
    for i, chunk in enumerate(chunks):
        chunk = chunk.strip()
        if not chunk:
            continue
        
        description = "" 
        
        # Handle Preface
        if i == 0:
            sec_type = "ส่วนนำ"
            sec_name = "ส่วนนำ"
            
            # ดักจับเส้นคั่น _______________________
            split_parts = re.split(r'\)?\s*\n+_{3,}\s*\n+', chunk, maxsplit=1)
            if len(split_parts) > 1:
                chunk = split_parts[1].strip()
                
        else:
            sec_type = current_type
            
            lines = [line.strip() for line in chunk.split('\n') if line.strip()]
            if not lines:
                continue
                
            if current_type == 'ข้อ':
                # ใช้ Regex จับแยก (Group 1: ชื่อข้อ) และ (Group 2: ข้อความหลังข้อ)
                match = re.match(r'^(ข้อ\s+\d+)\s*(.*)', lines[0])
                if match:
                    sec_name = match.group(1).strip()
                    description = match.group(2).strip() 
                else:
                    sec_name = lines[0]
                    
            elif current_type == 'คำศัพท์':
                match = re.search(r'“([^”]+)”', lines[0])
                sec_name = match.group(1).strip() if match else lines[0].split('หมายความว่า')[0].strip().replace('“', '').replace('”', '')
                
                next_pattern = patterns[1][0] if len(patterns) > 1 else None
                intro_text = re.split(next_pattern, chunk)[0] if next_pattern else chunk
                
                if 'หมายความว่า' in intro_text:
                    parts = intro_text.split('หมายความว่า', 1)
                    description = parts[1].strip() 
            
            elif current_type in ['ข้อย่อย', 'อนุย่อย']:
                match = re.match(r'^(\([ก-ฮ\d]+\)|\d+\.)\s*(.*)', lines[0])
                if match:
                    sec_name = match.group(1).strip()
                    description = match.group(2).strip()
                else:
                    sec_name = lines[0].split()[0]
                    
            else: # สำหรับ ภาค, หมวด, ส่วน
                sec_name = lines[0]
                if len(lines) > 1:
                    possible_desc = lines[1]
                    if not re.match(r'^_+$', possible_desc) and not possible_desc.startswith('ข้อ'):
                        description = possible_desc
        
        subsections.append({
            'section_type': sec_type,
            'section': sec_name,
            'parent': current_parents,
            'description': description, 
            'raw': chunk,
            'subsection': process_hierarchy(chunk, patterns[1:], current_parents + [sec_name])
        })
        
    return subsections

In [13]:
with open(INPUT_FILE, 'r', encoding='utf-8') as file:
    text = file.read()

part_chunks = re.split(r'\n+(?=ภาค\s+\d+\s*\n)', text.strip())

reg_data = {
    'data': []
}

intro_patterns = [
    (r'\n+(?=ข้อ\s+\d+)', 'ข้อ'),
    (r'\n+(?=“[^”]+”\s*หมายความว่า)', 'คำศัพท์'),
    (r'\n+(?=\s*(?:\(\d+\)|\d+\.)\s*)', 'ข้อย่อย'), 
    (r'\n+(?=\s*\([ก-ฮ]\)\s*)', 'อนุย่อย')
]

main_patterns = [
    (r'\n+(?=หมวด\s+\d+\s*\n)', 'หมวด'),
    (r'\n+(?=ส่วนที่\s+\d+\s*\n)', 'ส่วน'),
    (r'\n+(?=ข้อ\s+\d+)', 'ข้อ'),
    (r'\n+(?=\s*(?:\(\d+\)|\d+\.)\s*)', 'ข้อย่อย'), 
    (r'\n+(?=\s*\([ก-ฮ]\)\s*)', 'อนุย่อย')
]

for i, part_chunk in enumerate(part_chunks):
    part_chunk = part_chunk.strip()
    lines = [line.strip() for line in part_chunk.split('\n') if line.strip()]
    if not lines:
        continue
        
    description = ""
    
    if i == 0:
        sec_type = "บทนำ"
        part_name = "บทนำและนิยามศัพท์"
        patterns_to_use = intro_patterns
        
        desc_parts = re.split(r'_{3,}', part_chunk, maxsplit=1)
        if len(desc_parts) > 1:
            description = desc_parts[0].strip()
            
    else:
        sec_type = "ภาค"
        part_name = lines[0]
        patterns_to_use = main_patterns
        
        if len(lines) > 1:
            possible_desc = lines[1]
            if not re.match(r'^_+$', possible_desc) and not possible_desc.startswith('หมวด') and not possible_desc.startswith('ข้อ'):
                description = possible_desc
                
    reg_data['data'].append({
        'section_type': sec_type,
        'section': part_name,
        'parent': [],
        'description': description, 
        'raw': part_chunk,
        'subsection': process_hierarchy(part_chunk, patterns_to_use, [part_name])
    })


In [14]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as json_file:
    json.dump(reg_data, json_file, ensure_ascii=False, indent=4)

print(f"Saved to {OUTPUT_FILE} !")

Saved to intemediate_data/sec_reg_structured.json !
